PART 2

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import concat, when, lit, col

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("Your Data? Our Data - Part 2").getOrCreate()

# Create schema for pokemon.csv programatically
schema = StructType([
    StructField("Pokemon Name", StringType(), True),
    StructField("Primary Power", StringType(), True),
    StructField("Secondary Power", StringType(), True),
    StructField("Evolved Name", StringType(), True),
])
# Load the data first
# We use the schema defined above. Header=True skips the original CSV header row.
pokemon_df = spark.read.csv("pokemon.csv", header=True, schema=schema)

# Print the schema tree
pokemon_df.printSchema()

# Data Preview
pokemon_df.show(5)

root
 |-- Pokemon Name: string (nullable = true)
 |-- Primary Power: string (nullable = true)
 |-- Secondary Power: string (nullable = true)
 |-- Evolved Name: string (nullable = true)

+------------+-------------+---------------+------------+
|Pokemon Name|Primary Power|Secondary Power|Evolved Name|
+------------+-------------+---------------+------------+
|   bulbasaur|        Grass|         Poison|     ivysaur|
|     ivysaur|        Grass|         Poison|    venusaur|
|    venusaur|        Grass|         Poison|        NULL|
|  charmander|         Fire|           NULL|  charmeleon|
|  charmeleon|         Fire|           NULL|   charizard|
+------------+-------------+---------------+------------+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import col, when, lit, concat

#Cleaning missing data by filling nulls with "None" so the data is complete
pokemon_df = pokemon_df.fillna("None")

#Create a second table to find the next evolution stage
EvoTemp = pokemon_df.select(col("Pokemon Name").alias("NameTemp"), col("Evolved Name").alias("EvolutionTemp"))

#Join tables to see the whole tables in one row
joined_df = pokemon_df.join(EvoTemp, pokemon_df["Evolved Name"] == EvoTemp["NameTemp"], "left")

#Make a list of all names that are already evolved
evolved = [row[0] for row in pokemon_df.select("Evolved Name").filter(col("Evolved Name") != "None").distinct().collect()]

#Filter out the evolved named to keep the original pokemon
remove_df = joined_df.filter(~col("Pokemon Name").isin(evolved))

#Combine evolution names into one string
cleaned_df = remove_df.withColumn("Evolved Name",
    when(col("EvolutionTemp") != "None",
         concat(col("Evolved Name"), lit(", "), col("EvolutionTemp")) )
    .when(col("Evolved Name") != "None",
         col("Evolved Name") )
    .otherwise("None")
)

#Pick the columns to print
final_result = cleaned_df.select("Pokemon Name", "Primary Power", "Secondary Power", "Evolved Name")
final_result.show(100)

+------------+-------------+---------------+--------------------+
|Pokemon Name|Primary Power|Secondary Power|        Evolved Name|
+------------+-------------+---------------+--------------------+
|   bulbasaur|        Grass|         Poison|   ivysaur, venusaur|
|  charmander|         Fire|           None|charmeleon, chari...|
|    squirtle|        Water|           None|wartortle, blastoise|
|    caterpie|          Bug|           None| metapod, butterfree|
|      weedle|          Bug|         Poison|    kakuna, beedrill|
|      pidgey|       Normal|         Flying|  pidgeotto, pidgeot|
|     rattata|       Normal|           None|            raticate|
|     spearow|       Normal|         Flying|              fearow|
|       ekans|       Poison|           None|               arbok|
|     pikachu|     Electric|           None|              raichu|
|   sandshrew|       Ground|           None|           sandslash|
|   nidoran-f|       Poison|           None| nidorina, nidoqueen|
|   nidora

In [9]:
from pyspark.sql.functions import count, desc, when, col
# Insight 1: Distribution of Primary Powers
insight_1 = cleaned_df.groupBy("Primary Power").agg(count("*").alias("Total_Count")).orderBy(desc("Total_Count"))
print("Insight 1: Top 5 Primary Power Types")
insight_1.show(5)

Insight 1: Top 5 Primary Power Types
+-------------+-----------+
|Primary Power|Total_Count|
+-------------+-----------+
|        Water|        109|
|       Normal|        100|
|        Grass|         76|
|          Bug|         68|
|      Psychic|         51|
+-------------+-----------+
only showing top 5 rows


In [10]:
from pyspark.sql.functions import count, desc, when, col
# INSIGHT 2: Single vs Dual Type Distribution
insight_2 = cleaned_df.withColumn("Type_Category",
    when(col("Secondary Power") != "None", "Dual-Type").otherwise("Single-Type")).groupBy("Type_Category").agg(count("*").alias("Count"))

print("Insight 2: Single vs Dual Type Distribution")
insight_2.show()

Insight 2: Single vs Dual Type Distribution
+-------------+-----+
|Type_Category|Count|
+-------------+-----+
|    Dual-Type|  393|
|  Single-Type|  384|
+-------------+-----+



In [11]:
# Finding unique Type 1 + Type 2 pairings
rare_combos = cleaned_df.filter(col("Secondary Power") != "None").groupBy("Primary Power", "Secondary Power").agg(count("*").alias("Combo_Count")).orderBy("Combo_Count")
print("Insight 3: Rarest Type Combinations")
rare_combos.show(10)

Insight 3: Rarest Type Combinations
+-------------+---------------+-----------+
|Primary Power|Secondary Power|Combo_Count|
+-------------+---------------+-----------+
|       Ground|          Steel|          1|
|         Fire|          Steel|          1|
|        Steel|         Ground|          1|
|         Rock|         Poison|          1|
|        Ghost|         Dragon|          1|
|        Steel|         Dragon|          1|
|       Dragon|       Electric|          1|
|        Ghost|          Fairy|          1|
|      Psychic|          Steel|          1|
|     Fighting|          Steel|          1|
+-------------+---------------+-----------+
only showing top 10 rows


In [14]:
#Saving in Parquet
parquet_path = "parquet"
cleaned_df.write.format("parquet").mode("overwrite").save(parquet_path)